You can download IB Gateway or TWS at the following link:

https://www.interactivebrokers.com/en/trading/ibgateway-latest.php
https://www.interactivebrokers.ie/en/trading/download-tws.php?p=stable

Both options are equivalent to run this notebook. However:
- TWS has a more friendly user interface (UI)
- IB Gateways is lighter (best for compute-heavy applications)

Important info:

The default socket ports for TWS are:
- Live Trading: 7496
- Paper Trading: 7497

For Gateway:
- Live Trading: 4001
- Paper Trading: 4002

API step-by-step guide: https://www.interactivebrokers.com/campus/ibkr-quant-news/interactive-brokers-python-api-native-a-step-by-step-guide/



In [12]:
from ibapi.client import *
from ibapi.wrapper import *
from ibapi.contract import Contract
from ibapi.order import Order
import datetime
import time
import threading
import pandas as pd
import random

port = 7497

class TestApp(EClient, EWrapper):
    def __init__(self):
        EClient.__init__(self, self)
        self.historical_data = []
        self.orderId = 0
        self.data_complete = False

    def nextValidId(self, orderId: OrderId):
        self.orderId = orderId
        print(f"Connected! Next valid order ID: {orderId}")

    def nextId(self):
        self.orderId += 1
        return self.orderId

    def error(self, reqId, errorCode, errorString, advancedOrderReject=""):
        print(f"reqId: {reqId}, errorCode: {errorCode}, errorString: {errorString}")
        if errorCode == 326:
            print("Client ID already in use. Please disconnect previous connection or restart TWS/Gateway.")

    def headTimestamp(self, reqId, headTimeStamp):
        print(headTimeStamp)
        print(datetime.datetime.fromtimestamp(int(headTimeStamp)))
        self.cancelHeadTimeStamp(reqId)
    
    def historicalData(self, reqId, bar):
        self.historical_data.append({
            'Date': bar.date,
            'Open': bar.open,
            'High': bar.high,
            'Low': bar.low,
            'Close': bar.close,
            'Volume': bar.volume
        })
    
    def historicalDataEnd(self, reqId, start, end):
        print(f"Historical data download complete. Start: {start}, End: {end}")
        self.data_complete = True

In [13]:
# Connect to IBKR (run this cell once)
app = TestApp()
client_id = random.randint(1, 9999)
print(f"Connecting with client ID: {client_id}")
app.connect("127.0.0.1", port, client_id)
threading.Thread(target=app.run, daemon=True).start()
time.sleep(2)

if app.isConnected():
    print("Successfully connected!")
else:
    print("Failed to connect. Make sure TWS/Gateway is running on port 7497")


Connecting with client ID: 448
reqId: -1, errorCode: 502, errorString: Couldn't connect to TWS. Confirm that "Enable ActiveX and Socket EClients" 
is enabled and connection port is the same as "Socket Port" on the 
TWS "Edit->Global Configuration...->API->Settings" menu. Live Trading ports: 
TWS: 7496; IB Gateway: 4001. Simulated Trading ports for new installations 
of version 954.1 or newer:  TWS: 7497; IB Gateway: 4002
Failed to connect. Make sure TWS/Gateway is running on port 7497
Failed to connect. Make sure TWS/Gateway is running on port 7497


In [3]:
# Request historical data for Google
mycontract = Contract()
mycontract.symbol = "GOOGL"
mycontract.secType = "STK"
mycontract.exchange = "SMART"
mycontract.currency = "USD"

app.historical_data = []
app.data_complete = False

print("Requesting historical data for GOOGL...")
app.reqHistoricalData(app.nextId(), mycontract, "", "1 Y", "1 day", "TRADES", 1, 1, False, [])

# Wait for data to download
timeout = 30
start_time = time.time()
while not app.data_complete and (time.time() - start_time) < timeout:
    time.sleep(1)

# Convert to DataFrame
if app.historical_data:
    df = pd.DataFrame(app.historical_data)
    df['Date'] = pd.to_datetime(df['Date'])
    print(f"\nTotal records retrieved: {len(df)}")
    print("\nFirst few rows:")
    print(df.head())
    print("\nLast few rows:")
    print(df.tail())
else:
    print("No data retrieved. Check the error messages above.")


Requesting historical data for GOOGL...

Total records retrieved: 250

First few rows:
        Date    Open    High     Low   Close  Volume
0 2024-12-04  171.13  174.91  171.06  174.37  136586
1 2024-12-05  175.69  176.06  172.33  172.64  118850
2 2024-12-06  172.03  175.08  171.86  174.71   99946
3 2024-12-09  173.99  176.26  173.65  175.37  119268
4 2024-12-10  182.77  186.36  181.05  185.17  323761

Last few rows:
          Date    Open    High     Low   Close  Volume
245 2025-11-26  320.68  324.50  316.79  319.95  253620
246 2025-11-28  323.41  326.85  316.79  320.18  152893
247 2025-12-01  317.70  319.85  313.89  314.89  170087
248 2025-12-02  316.74  318.38  313.91  315.81  150501
249 2025-12-03  315.89  321.52  314.10  321.42  118551


## Free Continuous Quotes (Delayed)

IBKR lets every account stream **15-20 minute delayed quotes** for the exchanges you already have permissions for (typically NYSE/AMEX). Use a symbol listed on one of those venues—e.g. IBM, GE, XOM—when you just need a continuous feed for prototyping without paying for live market data.

In [ ]:
class DelayedStream(EClient, EWrapper):
    def __init__(self):
        EClient.__init__(self, self)
        self.orderId = 0
        self.price_data = []

    def nextValidId(self, orderId: OrderId):
        self.orderId = orderId
        print("Connected to delayed feed")

    def tickPrice(self, reqId, tickType, price, attrib):
        if tickType not in (66, 67, 68):  # delayed bid, ask, last
            return
        timestamp = datetime.datetime.now()
        label = {66: "Bid", 67: "Ask", 68: "Last"}[tickType]
        print(f"[{timestamp:%H:%M:%S}] {label}: ${price:.2f}")
        self.price_data.append({"timestamp": timestamp, "label": label, "price": price})

    def error(self, reqId, errorCode, errorString, advancedOrderReject=""):
        if errorCode in (2104, 2106, 2158):  # connection info
            return
        print(f"Error {errorCode}: {errorString}")


def start_delayed_stream(symbol="IBM", primary_exchange="NYSE"):
    """Start delayed streaming for a symbol listed on an exchange you have delayed permissions for."""
    if "delayed_app" in globals():
        try:
            delayed_app.disconnect()
            time.sleep(1)
        except Exception:
            pass
    app_instance = DelayedStream()
    client_id = random.randint(1, 9999)
    app_instance.connect("127.0.0.1", port, client_id)
    threading.Thread(target=app_instance.run, daemon=True).start()
    time.sleep(2)

    if not app_instance.isConnected():
        print("Failed to connect. Ensure TWS/Gateway is running on port 7497.")
        return None

    app_instance.reqMarketDataType(3)  # 3 = delayed

    contract = Contract()
    contract.symbol = symbol
    contract.secType = "STK"
    contract.exchange = "SMART"
    contract.currency = "USD"
    if primary_exchange:
        contract.primaryExchange = primary_exchange

    app_instance.reqMktData(1, contract, "", False, False, [])
    print(f"Streaming delayed quotes for {symbol} ({primary_exchange}). Run the next cell to stop.")
    return app_instance


delayed_app = start_delayed_stream()

Connected to delayed feed
Streaming delayed quotes for IBM (NYSE). Run the next cell to stop.


Error 10089: Requested market data requires additional subscription for API. See link in 'Market Data Connections' dialog for more details.IBM NYSE/TOP/ALL
Error 300: Can't find EId with tickerId:1


In [5]:
def stop_delayed_stream():
    if "delayed_app" not in globals() or delayed_app is None:
        print("No delayed stream to stop.")
        return

    try:
        delayed_app.cancelMktData(1)
        time.sleep(1)
        delayed_app.disconnect()
        print("Stopped delayed stream and disconnected")
        if delayed_app.price_data:
            df_delayed = pd.DataFrame(delayed_app.price_data)
            print(f"Collected {len(df_delayed)} delayed quotes. Last 5:")
            print(df_delayed.tail())
            return df_delayed
    finally:
        globals()["delayed_app"] = None


# Run after you are done streaming
df_delayed = stop_delayed_stream()

Stopped delayed stream and disconnected


## View Account Summary & Positions

Use the helper below to fetch balances (cash, buying power, PnL, etc.) plus any open positions into pandas DataFrames. Make sure TWS/Gateway is connected first.

In [11]:
class AccountWatcher(EClient, EWrapper):
    def __init__(self):
        EClient.__init__(self, self)
        self.account_rows = []
        self.positions = []
        self.account_done = threading.Event()
        self.positions_done = threading.Event()

    def nextValidId(self, orderId: OrderId):
        self.orderId = orderId
        print(f"Connected for account snapshot (nextId={orderId})")

    def accountSummary(self, reqId, account, tag, value, currency):
        self.account_rows.append({
            "account": account,
            "tag": tag,
            "value": float(value) if value not in ("", None) else None,
            "currency": currency,
        })

    def accountSummaryEnd(self, reqId):
        print("Account summary complete")
        self.account_done.set()

    def position(self, account, contract, position, avgCost):
        self.positions.append({
            "account": account,
            "symbol": contract.symbol,
            "secType": contract.secType,
            "position": position,
            "avgCost": avgCost,
            "currency": contract.currency,
        })

    def positionEnd(self):
        print("Positions snapshot complete")
        self.positions_done.set()

    def error(self, reqId, errorCode, errorString, advancedOrderReject=""):
        if errorCode in (2104, 2106, 2158):
            return
        print(f"Error {errorCode}: {errorString}")


def get_account_snapshot(tags=None, timeout=5):
    """Return (account_df, positions_df) using delayed-friendly snapshot requests."""
    tags = tags or [
        "TotalCashValue",
        "AvailableFunds",
        "BuyingPower",
        "NetLiquidation",
        "UnrealizedPnL",
        "RealizedPnL",
    ]

    watcher = AccountWatcher()
    client_id = random.randint(1, 9999)
    watcher.connect("127.0.0.1", port, client_id)
    threading.Thread(target=watcher.run, daemon=True).start()
    time.sleep(2)

    if not watcher.isConnected():
        print("Unable to connect for account snapshot. Check TWS/Gateway.")
        return None, None

    req_id = random.randint(1000, 9000)
    watcher.reqAccountSummary(req_id, "All", ",".join(tags))
    watcher.account_done.wait(timeout=timeout)
    watcher.reqPositions()
    watcher.positions_done.wait(timeout=timeout)
    watcher.disconnect()

    account_df = pd.DataFrame(watcher.account_rows)
    positions_df = pd.DataFrame(watcher.positions)
    return account_df, positions_df


account_df, positions_df = get_account_snapshot()

if account_df is not None and not account_df.empty:
    pivot = account_df.pivot_table(index="tag", values="value", aggfunc="first")
    print("\nAccount Summary:")
    display(pivot)
else:
    print("No account summary returned")

if positions_df is not None and not positions_df.empty:
    print("\nOpen Positions:")
    display(positions_df)
else:
    print("No open positions (or data unavailable)")

Connected for account snapshot (nextId=1)
Account summary complete
Positions snapshot complete

Account Summary:
Account summary complete
Positions snapshot complete

Account Summary:


,value
tag,
AvailableFunds,999996.97
BuyingPower,3999987.86
NetLiquidation,999997.08
TotalCashValue,999997.08



Open Positions:


,account,symbol,secType,position,avgCost,currency
0,DUO894372,IBM,STK,0.0,0.0,USD


## Placing Trades (Long, Sell, Short)

High-level workflow per order:
1. Ensure TWS/Gateway is running in Paper mode and API connections are allowed.
2. Define a `Contract` for the stock (symbol, secType `STK`, exchange `SMART`, currency `USD`).
3. Submit a `MARKET` order with `action` set to `BUY` (go long), `SELL` (close or reduce), or `SELL` with a currently flat account to establish a short (IBKR interprets this as shorting if you do not already hold shares).
4. Monitor fills through the callback events (`orderStatus`, `execDetails`) or by polling positions.

**Architecture note:** for small notebooks it is common to keep lightweight helper classes/functions (as below) so each action remains explicit. In larger projects, you would wrap all trade, account, and data logic inside a single controller/service class (or even separate modules) to centralize connection management and logging.

In [7]:
class TradeApp(EClient, EWrapper):
    def __init__(self):
        EClient.__init__(self, self)
        self.next_order_id = None
        self.ready = threading.Event()
        self.completed = threading.Event()
        self.status_log = []

    def nextValidId(self, orderId: OrderId):
        self.next_order_id = orderId
        print(f"Order connection ready (next valid id = {orderId})")
        self.ready.set()

    def orderStatus(self, orderId, status, filled, remaining, avgFillPrice,
                    permId, parentId, lastFillPrice, clientId, whyHeld, mktCapPrice):
        msg = {
            "orderId": orderId,
            "status": status,
            "filled": filled,
            "remaining": remaining,
            "avgFillPrice": avgFillPrice,
            "lastFillPrice": lastFillPrice,
        }
        self.status_log.append(msg)
        print(f"Order {orderId}: {status} (filled {filled}, remaining {remaining})")
        if status.lower() in ("filled", "cancelled", "inactive"):
            self.completed.set()

    def error(self, reqId, errorCode, errorString, advancedOrderReject=""):
        if errorCode in (2104, 2106, 2158):
            return
        print(f"Error {errorCode}: {errorString}")


def build_stock_contract(symbol, primary_exchange="SMART"):
    contract = Contract()
    contract.symbol = symbol
    contract.secType = "STK"
    contract.exchange = "SMART"
    contract.currency = "USD"
    contract.primaryExchange = primary_exchange
    return contract


def make_market_order(action, quantity):
    order = Order()
    order.action = action
    order.orderType = "MKT"
    order.totalQuantity = quantity
    order.transmit = True
    order.eTradeOnly = False  # Explicitly disable unsupported E*TRADE flag
    order.firmQuoteOnly = False
    return order


def submit_market_order(symbol, quantity, action="BUY", primary_exchange="SMART", timeout=10):
    app = TradeApp()
    client_id = random.randint(1, 9999)
    app.connect("127.0.0.1", port, client_id)
    threading.Thread(target=app.run, daemon=True).start()

    if not app.ready.wait(timeout=3):
        print("Timed out waiting for next valid order id")
        return None

    contract = build_stock_contract(symbol, primary_exchange)
    order = make_market_order(action, quantity)
    order_id = app.next_order_id
    print(f"Placing {action} market order for {quantity} {symbol}")
    app.placeOrder(order_id, contract, order)

    app.completed.wait(timeout=timeout)
    app.disconnect()
    if not app.status_log:
        print("No order status callbacks were received. Ensure API trading is enabled (Global Configuration → API → Settings) and check TWS for warnings.")
    return app.status_log


def go_long(symbol, quantity, primary_exchange="SMART"):
    """Buy shares (opens or adds to a long position)."""
    return submit_market_order(symbol, quantity, action="BUY", primary_exchange=primary_exchange)


def sell_position(symbol, quantity, primary_exchange="SMART"):
    """Sell shares you already own (partial or full)."""
    return submit_market_order(symbol, quantity, action="SELL", primary_exchange=primary_exchange)


def go_short(symbol, quantity, primary_exchange="SMART"):
    """Enter a short position (SELL when you do not currently hold shares)."""
    return submit_market_order(symbol, quantity, action="SELL", primary_exchange=primary_exchange)


# Example usage (uncomment to try in paper account)
# go_long("IBM", 10)
# sell_position("IBM", 5)
# go_short("IBM", 5)  # Ensure your account has short permissions

In [10]:
sell_position("IBM", 5)

Order connection ready (next valid id = 1)
Placing SELL market order for 5 IBM
Order 1: PreSubmitted (filled 0.0, remaining 5.0)
Order 1: PreSubmitted (filled 0.0, remaining 5.0)
Order 1: Filled (filled 5.0, remaining 0.0)


[{'orderId': 1,
  'status': 'PreSubmitted',
  'filled': 0.0,
  'remaining': 5.0,
  'avgFillPrice': 0.0,
  'lastFillPrice': 0.0},
 {'orderId': 1,
  'status': 'PreSubmitted',
  'filled': 0.0,
  'remaining': 5.0,
  'avgFillPrice': 0.0,
  'lastFillPrice': 0.0},
 {'orderId': 1,
  'status': 'Filled',
  'filled': 5.0,
  'remaining': 0.0,
  'avgFillPrice': 302.65,
  'lastFillPrice': 302.65}]

In [9]:
sell_position("IBM", 5)

Order connection ready (next valid id = 1)
Placing SELL market order for 5 IBM
Order 1: PreSubmitted (filled 0.0, remaining 5.0)
Order 1: Submitted (filled 0.0, remaining 5.0)
Order 1: Filled (filled 5.0, remaining 0.0)
Order 1: PreSubmitted (filled 0.0, remaining 5.0)
Order 1: Submitted (filled 0.0, remaining 5.0)
Order 1: Filled (filled 5.0, remaining 0.0)


[{'orderId': 1,
  'status': 'PreSubmitted',
  'filled': 0.0,
  'remaining': 5.0,
  'avgFillPrice': 0.0,
  'lastFillPrice': 0.0},
 {'orderId': 1,
  'status': 'Submitted',
  'filled': 0.0,
  'remaining': 5.0,
  'avgFillPrice': 0.0,
  'lastFillPrice': 0.0},
 {'orderId': 1,
  'status': 'Filled',
  'filled': 5.0,
  'remaining': 0.0,
  'avgFillPrice': 302.52,
  'lastFillPrice': 302.52}]

In [9]:
account_df, positions_df = get_account_snapshot()

if account_df is not None and not account_df.empty:
    pivot = account_df.pivot_table(index="tag", values="value", aggfunc="first")
    print("\nAccount Summary:")
    display(pivot)
else:
    print("No account summary returned")

if positions_df is not None and not positions_df.empty:
    print("\nOpen Positions:")
    display(positions_df)
else:
    print("No open positions (or data unavailable)")

Connected for account snapshot (nextId=1)
Account summary complete
Positions snapshot complete

Account Summary:
Account summary complete
Positions snapshot complete

Account Summary:


,value
tag,
AvailableFunds,998493.06
BuyingPower,3993972.23
NetLiquidation,999996.77
TotalCashValue,995451.77



Open Positions:


,account,symbol,secType,position,avgCost,currency
0,DUO894372,IBM,STK,20.0,302.755026,USD


## ⚠️ Market Data Subscription Required

**Error 10089** means you need a market data subscription for real-time data. 

**Solutions:**

1. **Use Delayed Data (Free)** - 15-20 minute delayed quotes (see cell below)
2. **Subscribe to Market Data** - In TWS/Gateway: 
   - Go to Account → Market Data Subscriptions
   - Subscribe to "US Securities Snapshot and Futures Value Bundle" (~$1-4.50/month)
   - Or "US Equity and Options Add-On Streaming Bundle" for real-time

3. **Use Real-Time Bars (Free for some contracts)** - 5-second bars instead of tick data

In [10]:
# Disconnect when done (run this at the end)
if app.isConnected():
    app.disconnect()
    print("Disconnected from IBKR")
else:
    print("Already disconnected")


Disconnected from IBKR


## For .py files: Always disconnect

When using this in a Python script (`.py` file), always wrap your code with connection and disconnection:

```python
if __name__ == "__main__":
    app = TestApp()
    app.connect("127.0.0.1", 7497, random.randint(1, 9999))
    threading.Thread(target=app.run, daemon=True).start()
    time.sleep(2)
    
    try:
        # Your trading logic here
        # Request data, place orders, etc.
        pass
    finally:
        # Always disconnect, even if an error occurs
        app.disconnect()
        print("Disconnected from IBKR")
```

This ensures proper cleanup and prevents "client ID already in use" errors on subsequent runs.
